In [13]:
import pandas as pd
import numpy as np

# Clases propias
from etl import Dataloader
from feature_engineer import add_features
from train import Train
from train_with_mlflow import TrainWithMLflow

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
from xgboost import XGBClassifier

# MLflow
import mlflow


ETL

In [14]:
loader = Dataloader(
    path_to_save=r"C:/Users/Valentina Molina/Documents/Repositorios/Proyecto_Final_MLOps/data/PS_20174392719_1491204439457_log.csv",
    n_samples=50000
)
df = loader.load_data()
loader.drop_name_columns()
df = loader.df.copy()

print(df.shape)
df.head()

(50000, 8)


,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud
0,1,PAYMENT,9839.64,170136.0,160296.36,0.0,0.0,0
1,1,PAYMENT,1864.28,21249.0,19384.72,0.0,0.0,0
2,1,TRANSFER,181.00,181.0,0.00,0.0,0.0,1
3,1,CASH_OUT,181.00,181.0,0.00,21182.0,0.0,1
4,1,PAYMENT,11668.14,41554.0,29885.86,0.0,0.0,0


Feature Engineering

In [15]:
df = add_features(df)
df.head()

,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,diff_old_new_orig,diff_old_new_dest,amount_to_orig_balance,amount_to_dest_balance
0,1,PAYMENT,9839.64,170136.0,160296.36,0.0,0.0,0,9839.64,0.0,0.057834,9839.640000
1,1,PAYMENT,1864.28,21249.0,19384.72,0.0,0.0,0,1864.28,0.0,0.087731,1864.280000
2,1,TRANSFER,181.00,181.0,0.00,0.0,0.0,1,181.00,0.0,0.994505,181.000000
3,1,CASH_OUT,181.00,181.0,0.00,21182.0,0.0,1,181.00,21182.0,0.994505,0.008545
4,1,PAYMENT,11668.14,41554.0,29885.86,0.0,0.0,0,11668.14,0.0,0.280788,11668.140000


Definir variables

In [16]:
numeric_features = [
    'step', 'amount',
    'oldbalanceOrg', 'newbalanceOrig',
    'oldbalanceDest', 'newbalanceDest',
    'diff_old_new_orig', 'diff_old_new_dest',
    'amount_to_orig_balance', 'amount_to_dest_balance'
]

categorical_features = ['type']
target_column = 'isFraud'
test_size = 0.2


Step 1: Modelando sin MLflow

In [17]:
# Logistic Regression
model = LogisticRegression(max_iter=1000, random_state=42)
trainer = Train(df, numeric_features, categorical_features, target_column, model, test_size)
pipeline = trainer.train()
print("Modelo Logistic Regression entrenado sin MLflow ✅")

# Random Forest
model = RandomForestClassifier(n_estimators=100, random_state=42)
trainer = Train(df, numeric_features, categorical_features, target_column, model, test_size)
pipeline = trainer.train()
print("Modelo RandomForest entrenado sin MLflow ✅")


2025/09/28 08:42:56 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'f8124f531838480589eebf9329122cee', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow


🏃 View run gaudy-skunk-915 at: http://127.0.0.1:5000/#/experiments/991532212819800877/runs/f8124f531838480589eebf9329122cee
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/991532212819800877
Modelo Logistic Regression entrenado sin MLflow ✅


2025/09/28 08:43:28 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '5bf8251add66456e8777d71500979c85', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow


🏃 View run hilarious-koi-798 at: http://127.0.0.1:5000/#/experiments/991532212819800877/runs/5bf8251add66456e8777d71500979c85
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/991532212819800877
Modelo RandomForest entrenado sin MLflow ✅


Step 2: Modelando con MLflow (Logistic + RandomForest + LightGBM)

In [18]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("fraude-modelos")

mlflow.sklearn.autolog()

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb

modelos = [
    ("LogisticRegression", LogisticRegression(max_iter=1000, random_state=42)),
    ("RandomForest", RandomForestClassifier(n_estimators=100, random_state=42)),
    ("LightGBM", lgb.LGBMClassifier(random_state=42))
]

In [19]:
resultados = {}
for nombre, modelo in modelos:
    print(f"\nEntrenando {nombre} con MLflow...")
    trainer = TrainWithMLflow(df, numeric_features, categorical_features, target_column, modelo, test_size)
    pipeline, run_id = trainer.train()
    resultados[nombre] = run_id

resultados


Entrenando LogisticRegression con MLflow...
MLflow Run ID: cf6aaed3d22447d39bf47b902a07d0d1
Tracking URI: http://127.0.0.1:5000
Train Accuracy: 0.9984
Test Accuracy: 0.9978
🏃 View run nervous-squid-274 at: http://127.0.0.1:5000/#/experiments/126333943536851071/runs/cf6aaed3d22447d39bf47b902a07d0d1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/126333943536851071

Entrenando RandomForest con MLflow...
MLflow Run ID: ff97affd109f4bef8ab41be30d2d378d
Tracking URI: http://127.0.0.1:5000
Train Accuracy: 1.0000
Test Accuracy: 0.9997
🏃 View run unique-finch-884 at: http://127.0.0.1:5000/#/experiments/126333943536851071/runs/ff97affd109f4bef8ab41be30d2d378d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/126333943536851071

Entrenando LightGBM con MLflow...
[LightGBM] [Info] Number of positive: 80, number of negative: 39920
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.018087 seconds.
You can set `force_col_wise=true` to remove th

{'LogisticRegression': 'cf6aaed3d22447d39bf47b902a07d0d1',
 'RandomForest': 'ff97affd109f4bef8ab41be30d2d378d',
 'LightGBM': '0120eabc335d478bb5be957f4165331c'}

Modelo con XGBoost + MLflow

In [20]:
mlflow.set_experiment("fraude-xgboost")
mlflow.xgboost.autolog()

params_xgb = {
    "n_estimators": 100,
    "max_depth": 6,
    "learning_rate": 0.1,
    "subsample": 0.8
}

model_xgb = XGBClassifier(**params_xgb, random_state=42, use_label_encoder=False, eval_metric="logloss")

trainer = TrainWithMLflow(df, numeric_features, categorical_features, target_column, model_xgb, test_size, params_xgb, mlflow)
pipeline, run_id = trainer.train()
print("Modelo XGBoost entrenado con MLflow ✅", run_id)


MLflow Run ID: 01ff03c6b38c4f81b2c34114d0cc8b20
Tracking URI: http://127.0.0.1:5000
Train Accuracy: 1.0000
Test Accuracy: 0.9994
🏃 View run suave-squid-529 at: http://127.0.0.1:5000/#/experiments/670538311910718369/runs/01ff03c6b38c4f81b2c34114d0cc8b20
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/670538311910718369
Modelo XGBoost entrenado con MLflow ✅ 01ff03c6b38c4f81b2c34114d0cc8b20


Step 3: Modelo con MLflow + Optuna

In [21]:
from train_with_mlflow_optuna import TrainWithMLflowOptuna

mlflow.set_experiment("fraude-optuna")

param_distributions = {
    'n_estimators': ('int', 50, 200),
    'max_depth': ('int', 5, 30),
    'min_samples_split': ('int', 2, 10),
    'min_samples_leaf': ('int', 1, 5),
    'max_features': ('categorical', ['sqrt', 'log2', None])
}

trainer = TrainWithMLflowOptuna(
    df=df,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    target_column=target_column,
    model_class=RandomForestClassifier,
    test_size=0.2,
    n_trials=20,
    optimization_metric='f1',
    param_distributions=param_distributions,
    model_params={'random_state': 42},
    mlflow_setup=mlflow
)

best_pipeline, run_id, study = trainer.train()
print("Mejor modelo con Optuna:", run_id)


[I 2025-09-28 08:45:33,550] A new study created in memory with name: no-name-aad08937-2a2e-4aed-ad4f-d85d6128f531
2025/09/28 08:45:33 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '4b511d67f42d487582a16ce024e38a51', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow


🏃 View run chill-hawk-299 at: http://127.0.0.1:5000/#/experiments/991532212819800877/runs/4b511d67f42d487582a16ce024e38a51
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/991532212819800877


[I 2025-09-28 08:45:59,752] Trial 0 finished with value: 0.8888888888888888 and parameters: {'n_estimators': 110, 'max_depth': 27, 'min_samples_split': 6, 'min_samples_leaf': 3, 'max_features': 'log2'}. Best is trial 0 with value: 0.8888888888888888.
2025/09/28 08:45:59 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '9f2aa8d47d594f81a55f58ffeb6981b2', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow


🏃 View run crawling-bear-964 at: http://127.0.0.1:5000/#/experiments/991532212819800877/runs/9f2aa8d47d594f81a55f58ffeb6981b2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/991532212819800877


[I 2025-09-28 08:46:42,282] Trial 1 finished with value: 0.0 and parameters: {'n_estimators': 86, 'max_depth': 5, 'min_samples_split': 8, 'min_samples_leaf': 1, 'max_features': None}. Best is trial 0 with value: 0.8888888888888888.
2025/09/28 08:46:42 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'b09a5c9ef3624424b6f6ae6d2ac20461', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow


...

🏃 View run painted-swan-262 at: http://127.0.0.1:5000/#/experiments/991532212819800877/runs/76113106dac64a40b0c93caf9d6c6406
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/991532212819800877


[I 2025-09-28 08:58:41,877] Trial 18 finished with value: 0.918918918918919 and parameters: {'n_estimators': 178, 'max_depth': 20, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'sqrt'}. Best is trial 8 with value: 0.918918918918919.
2025/09/28 08:58:42 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'b3ee28c60e2847a9b3d7431d4e1ae58d', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow


🏃 View run clean-crow-709 at: http://127.0.0.1:5000/#/experiments/991532212819800877/runs/b3ee28c60e2847a9b3d7431d4e1ae58d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/991532212819800877


[I 2025-09-28 08:59:10,251] Trial 19 finished with value: 0.918918918918919 and parameters: {'n_estimators': 154, 'max_depth': 13, 'min_samples_split': 3, 'min_samples_leaf': 1, 'max_features': 'log2'}. Best is trial 8 with value: 0.918918918918919.
2025/09/28 08:59:10 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '80cc7c7c7db14aa0934e0762379402df', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow


🏃 View run righteous-hen-542 at: http://127.0.0.1:5000/#/experiments/991532212819800877/runs/80cc7c7c7db14aa0934e0762379402df
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/991532212819800877
Optuna best params: {'random_state': 42, 'n_estimators': 174, 'max_depth': 27, 'min_samples_split': 6, 'min_samples_leaf': 1, 'max_features': 'sqrt'}
MLflow run_id: a8d1afd94b964c5b96d3084277321b66
🏃 View run gaudy-deer-139 at: http://127.0.0.1:5000/#/experiments/991532212819800877/runs/a8d1afd94b964c5b96d3084277321b66
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/991532212819800877
Mejor modelo con Optuna: a8d1afd94b964c5b96d3084277321b66


In [ ]:
import pickle

with open('../pred-batch/models/best_model.pkl', 'wb') as f:

    pickle.dump(pipeline, f)